## 依赖导入

本例集合演示降维与因子分析：用 `sklearn.decomposition.PCA` 做主成分分析（标准化、贡献率、主成分回归、核主成分综合评价），用 `factor_analyzer.FactorAnalyzer` 做 varimax 旋转的因子分析与因子得分评价。统一导入 `numpy`、`pandas`、`statsmodels`、`zscore`、`PCA` 与 `FA`。

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from factor_analyzer import FactorAnalyzer as FA
from scipy.stats import zscore
from sklearn.decomposition import PCA

## 10.1  PCA 拟合与主成分系数

读取 `data/12_1.txt`，用 `zscore(ddof=1)` 标准化后 `PCA().fit`，输出特征值 `explained_variance_`、各主成分贡献率与主成分系数（每行一个主成分）。再按各主成分系数之和的符号统一方向（和为负则乘以 $-1$），输出调整后的主成分系数（源程序 ex12_1.py）。

In [ ]:
a = np.loadtxt("data/12_1.txt")
b = zscore(a, ddof=1)
md = PCA().fit(b)
print("特征值为：", md.explained_variance_)
print("各主成分贡献率：", md.explained_variance_ratio_)
xs1 = md.components_
print("各主成分系数:\n", xs1)
check = xs1.sum(axis=1, keepdims=True)
xs2 = xs1 * np.sign(check)
print("调整后的主成分系数：", xs2)

## 10.2  主成分回归

读取 `data/12_2.txt`，逐列求均值、标准差并标准化 `b`（去掉最后一列因变量），对其做 `PCA`，输出特征值、贡献率、主成分系数与累积贡献率。取前 3 个主成分得 `f`；分别用原始数据 `ols("y~x")` 与主成分 `ols("y~z")` 回归，再把主成分回归系数换算回原始变量（常数项 `xs40`、其他系数 `xs4`），并比较两种回归的残差方差 `mse_resid`（源程序 ex12_2.py）。

In [ ]:
a = np.loadtxt("data/12_2.txt")
mu = a.mean(axis=0)
s = a.std(axis=0, ddof=1)
b = (a - mu) / s
r = np.corrcoef(b[:, :-1].T)
md1 = PCA().fit(b[:, :-1])
print("特征值为：", md1.explained_variance_)
print("各主成分贡献率：", md1.explained_variance_ratio_)
xs = md1.components_
print("主成分系数：\n", np.round(xs, 4))
print("累积贡献率：", np.cumsum(md1.explained_variance_ratio_))
n = 3
f = b[:, :-1] @ (xs[:n, :].T)
d2 = {"y": a[:, -1], "x": a[:, :-1]}
md2 = sm.formula.ols("y~x", d2).fit()
d3 = {"y": a[:, -1], "z": f}
md3 = sm.formula.ols("y~z", d3).fit()
xs3 = md3.params
xs40 = xs3[0] - sum(xs3[1:] @ xs[:n, :] * mu[:-1] / s[:-1])
xs4 = xs3[1:] @ xs[:n, :] / s[:-1]
print("回归方程的常数项：", round(xs40, 4))
print("回归方程的其他系数：", np.round(xs4, 4))
print("直接回归的残差方差：", md2.mse_resid)
print("主成分回归的残差方差：", md3.mse_resid)

## 10.3  主成分与核主成分综合评价

读取 `data/12_3.xlsx` 标准化。上半做普通 `PCA`：取前 4 个主成分，按贡献率加权得综合得分 `g1`，用 `argsort` 给出从大到小的名次。下半为核主成分分析：构造多项式核 $K=(1+c_i\cdot c_j)^3$ 并中心化得 `Kw`，对其做 `PCA`、取前 2 主成分加权得 `g2` 并输出名次（源程序 ex12_3.py）。

In [ ]:
a = pd.read_excel("data/12_3.xlsx", header=None)
b = a.values
c = zscore(b, ddof=1)
md1 = PCA().fit(c)
print("特征值为：", md1.explained_variance_)
r1 = md1.explained_variance_ratio_
print("各主成分贡献率：", r1)
xs1 = md1.components_
print("主成分系数：\n", np.round(xs1, 4))
print("累积贡献率：", np.cumsum(r1))
n1 = 4
df1 = c @ (xs1[:n1, :].T)
g1 = df1 @ r1[:n1]
print("主成分评价得分", np.round(g1, 4))
ind1 = np.argsort(-g1)
ind11 = np.zeros(15)
ind11[ind1] = np.arange(1, 16)
print("排序结果：", ind11)
print("---------------")
n = b.shape[0]
K = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        K[i, j] = (1 + c[i] @ c[j]) ** 3
In = np.ones((n, n)) / n
Kw = K - K @ In - In @ K + In @ K @ In
md2 = PCA().fit(Kw)
print("特征值为：", md2.explained_variance_)
r2 = md2.explained_variance_ratio_
print("各主成分贡献率：", r2)
xs2 = md2.components_
print("主成分系数：\n", np.round(xs2, 4))
print("累积贡献率：", np.cumsum(r2))
n2 = 2
df2 = Kw @ (xs2[:n2, :].T)
g2 = df2 @ r2[:n2]
print("核主成分评价得分：", np.round(g2))
ind2 = np.argsort(-g2)
ind22 = np.zeros(15)
ind22[ind2] = np.arange(1, 16)
print("排序结果：", ind22)

## 10.4  因子分析主成分解与载荷矩阵

给定相关系数阵 `r`，用 `np.linalg.eig` 求特征值/特征向量，按广播 $A_0=\text{vec}\cdot\sqrt{\text{val}}$ 得载荷矩阵；由 `input` 读入公共因子个数 `num`，截取载荷阵 `A`，计算各变量的信息贡献（逐列平方和 `Ac`）与共同度（逐行平方和 `Ar`）（源程序 ex12_4.py）。

In [ ]:
r = np.array([[1, 1 / 5, -1 / 5], [1 / 5, 1, -2 / 5], [-1 / 5, -2 / 5, 1]])
val, vec = np.linalg.eig(r)
A0 = vec * np.sqrt(val)
print("特征值:", val, "\n载荷矩阵：\n", A0, "\n----------")
num = int(input("请输入选择公共因子的个数："))
A = A0[:, :num]
Ac = np.sum(A**2, axis=0)
Ar = np.sum(A**2, axis=1)
print("对x的贡献为：", Ac)
print("共同度为：", Ar)

## 10.5  因子分析与 varimax 旋转（两种载荷提取）

读取 `data/12_5_1.txt` 标准化、求相关阵 `r` 及其特征值贡献率（`sorted(rate, reverse=True)` 降序打印）。用 `FA(3, rotation="varimax")` 拟合，以 `fa.loadings_` 取载荷矩阵 `A`，计算信息贡献 `gx`、特殊方差 `s2`、因子得分系数与得分 `df`，按 $pj = df\cdot gx/\sum gx$ 得综合评价值并排名，最后把 `r`、`df`、`pj` 写入 `data/12_5_2.xlsx`（源程序 ex12_5_1.py）。

In [ ]:
c = np.loadtxt("data/12_5_1.txt")
d = zscore(c, ddof=1)
r = np.corrcoef(d.T)
val, vec = np.linalg.eig(r)
cs = np.cumsum(val)
rate = val / cs[-1]
srate = sorted(rate, reverse=True)
print("特征值为：", val, "\n贡献率为：", srate)
fa = FA(3, rotation="varimax")
fa.fit(d)
A = fa.loadings_
gx = np.sum(A**2, axis=0)
s2 = 1 - np.sum(A**2, axis=1)
ss = np.linalg.inv(np.diag(s2))
f = ss @ A @ np.linalg.inv(A.T @ ss @ A)
df = d @ f
pj = df @ gx / sum(gx)
print("载荷矩阵为：\n", np.round(A, 4))
print("特殊方差为：", np.round(s2, 4))
print("各因子的方差贡献：", np.round(gx, 4))
print("评价值为：\n", pj)
ind0 = np.argsort(-pj)
ind = np.zeros(17)
ind[ind0] = np.arange(1, 18)
print("排名次序为：", ind)
F = pd.ExcelWriter("data/12_5_2.xlsx")
pd.DataFrame(r).to_excel(F)
pd.DataFrame(df).to_excel(F, "Sheet2", index=False)
pd.DataFrame(pj).to_excel(F, "Sheet3", index=False)
F.save()

同一数据的第二种写法：改用 `fa.components_.T` 作为载荷矩阵 `A`，其余信息贡献 `gx`、特殊方差 `s2`、因子得分 `df` 与综合评价值 `pj` 的计算相同，输出载荷矩阵、特殊方差并计算得分 `pj`（源程序 ex12_5.py）。

In [ ]:
c = np.loadtxt("data/12_5_1.txt")
d = zscore(c, ddof=1)
r = np.corrcoef(d.T)
val, vec = np.linalg.eig(r)
cs = np.cumsum(val)
rate = val / cs[-1]
print("特征值为：", val, "\n贡献率为：", rate)
fa = FA(3, rotation="varimax")
fa.fit(d)
A = fa.components_.T
gx = np.sum(A**2, axis=0)
s2 = 1 - np.sum(A**2, axis=1)
ss = np.linalg.inv(np.diag(s2))
df = d @ ss @ A @ np.linalg.inv(A.T @ ss @ A)
print("载荷矩阵为：\n", A)
print("特殊方差为：\n", s2)
pj = df @ gx / sum(gx)